# Session 6 — Extension: how amino-acid class changes and region shape PTEN abundance

**Question.** What are the relationships between the *class change* of a substitution (wild-type residue class → mutant residue class) and the *region* of PTEN it falls in, on abundance and on the change in abundance relative to wild type (WT)?

**Hypothesis.**
1. The more different a variant's side chain is from WT, the greater the change in abundance. Variants that change class should have larger aggregate effects than same-class variants.
2. The phosphatase and C2 domains show the largest changes.
3. Charged → hydrophobic substitutions have the strongest effect in the phosphatase domain.

**Classes** come from `aminoacids-pic.png` (the chart used in class): charged (Arg, His, Lys, Asp, Glu), polar uncharged (Ser, Thr, Asn, Gln), special cases (Cys, Gly, Pro), hydrophobic (Ala, Val, Ile, Leu, Met, Phe, Tyr, Trp). 4 WT classes × 4 mutant classes = **16** class changes (12 change class, 4 stay in the same class).

**Plan (figure names are also the PDF names).**
- Section 2, raw abundance score: `hm_abund_0` (all sites × all mutant residues), `hm_abund_1` (observed wt>mut substitutions), `hm_abund_2` (the same rows grouped by class change, not aggregated), `hm_abund_3` (median per class change × 25-site window).
- Section 3, log ratio to WT (`log2(score / 1)`): `hm_diff_0` to `hm_diff_3`, same layouts.
- Section 4, region summaries: raw first, refined, then the log ratio.
- Section 5, the test: a permutation test of the charged → hydrophobic prediction in the phosphatase domain, plus an exploratory screen of all class change × region cells.

**Scope notes.** The assay measures *abundance* only (not activity or pathogenicity). Scores are already scaled so that WT = 1 and nonsense ≈ 0, so "relative to WT" means `score / 1`. All figures are saved as PDFs in `data/my_data/pdfs/`.

## 0. Setup

In [ ]:
# Setup: load the libraries this notebook uses.
import numpy as np                      # arrays and math (np.log2, np.median, random numbers), like base R vectors/math
import pandas as pd                     # tables (DataFrames), like R data frames / tibbles
import matplotlib.pyplot as plt         # plotting; plt.subplots() makes the canvas, like setting up a ggplot panel
import seaborn as sns                   # statistical plots built on matplotlib (boxplots, palettes)
from matplotlib.colors import TwoSlopeNorm, ListedColormap   # a color scale with a chosen center; a fixed list of colors
from matplotlib.patches import Patch    # colored squares for hand-made legends
from pathlib import Path                # file paths as objects; Path / "folder" joins them, like file.path()

### Functions
Defined once, here. Contracts are in the docstrings, and the known-answer tests are in sections 1 and 5.

In [ ]:
# All the functions for this notebook in one place. Some are copied from Sessions 3-5; the new ones are
# log2_ratio, make_grid, plot_site_heatmap, draw_window_heatmap, save_pdf, plot_class_by_region, label_color, cell_labels and the two permutation tests.

# 3-letter amino acid code -> 1-letter code (hgvs_pro labels use 3 letters, the FASTA file uses 1 letter)
letter = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G",
          "His": "H", "Ile": "I", "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S",
          "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V"}
one_to_three = {v: k for k, v in letter.items()}   # dict comprehension: loop over the pairs and flip key and value

# Side-chain class of each amino acid, following aminoacids-pic.png.
# A dict keeps insertion order, so list(aa_class) is the chart order: charged, polar, special, hydrophobic.
aa_class = {}
for class_name, members in [("charged", ["Arg", "His", "Lys", "Asp", "Glu"]),
                            ("polar", ["Ser", "Thr", "Asn", "Gln"]),
                            ("special", ["Cys", "Gly", "Pro"]),
                            ("hydrophobic", ["Ala", "Val", "Ile", "Leu", "Met", "Phe", "Tyr", "Trp"])]:
    for aa in members:
        aa_class[aa] = class_name

class_order = ["charged", "polar", "special", "hydrophobic"]
aa_order = list(aa_class)   # the 20 residues, grouped by class
# All 16 class changes as "wt_class->mut_class". A list comprehension with two for-clauses is a nested loop:
# the first for is the outer loop, so each WT class forms a block of 4 consecutive labels.
change_order = [f"{wt}->{mut}" for wt in class_order for mut in class_order]


def read_fasta(path):
    """Return the protein sequence in a FASTA file as one string."""
    lines = path.read_text().splitlines()   # read the whole file, split into a list of lines
    return "".join(line.strip() for line in lines if not line.startswith(">"))   # drop the ">" header, glue the rest


def get_checked_missense(table, ref_seq):
    """Return the missense rows of table, with new columns wt, pos, and mut.

    Raises ValueError if a label cannot be parsed, is not a real substitution, sits outside the protein,
    has a WT residue that does not match ref_seq, or is duplicated. Missing scores are kept as NaN.
    """
    df = table[table["record_type"] == "missense"].copy()   # boolean mask, like filter() in dplyr; copy() avoids edit warnings

    # "p.His75Asp" -> wt "His", site 75, mutant "Asp". str.extract uses one regex group per new column.
    parts = df["hgvs_pro"].str.extract(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")
    if parts[0].isna().any():
        raise ValueError("unparseable label(s): " + str(df.loc[parts[0].isna(), "hgvs_pro"].tolist()[:5]))
    df["wt"], df["pos"], df["mut"] = parts[0], parts[1].astype(int), parts[2]   # tuple unpacking: three assignments in one line

    if not df["mut"].isin(list(letter)).all() or (df["mut"] == df["wt"]).any():
        raise ValueError("not a missense substitution")
    if not df["pos"].between(1, len(ref_seq)).all():   # between() is inclusive on both ends
        raise ValueError("site outside the protein")

    # sites count from 1 but Python indexes from 0, so label the reference letters 1, 2, 3, ...
    ref_by_site = pd.Series(list(ref_seq), index=range(1, len(ref_seq) + 1))
    if (df["wt"].map(letter) != df["pos"].map(ref_by_site)).any():   # .map(dict or Series) is a lookup, like match() in R
        raise ValueError("WT residue does not match the reference")
    if df["hgvs_pro"].duplicated().any():
        raise ValueError("duplicate labels")

    df["score"] = pd.to_numeric(df["score"])
    return df[["hgvs_pro", "wt", "pos", "mut", "score", "expts"]]   # double brackets = a list of columns -> a DataFrame


def make_site_region(regions, n_sites):
    """Region name for each site 1..n_sites; sites in no annotated region are "unannotated"."""
    site_region = pd.Series("unannotated", index=range(1, n_sites + 1))   # one value repeated over the index
    for _, r in regions.iterrows():   # iterrows() yields (row number, row); "_" is a throwaway name for the unused row number
        site_region.loc[r["start"]:r["end"]] = r["region"]   # .loc slices by label and INCLUDES both ends
    return site_region


def log2_ratio(scores):
    """log2(score / WT) where WT = 1, so log2(score). Scores <= 0 have no log and become NaN (not 0, not -inf).

    Works on a Series or a DataFrame: where() keeps the values that pass the test and sets the rest to NaN.
    """
    return np.log2(scores.where(scores > 0))


def make_grid(df, row_col, row_order, value_col, n_sites):
    """Wide table for a heatmap: one row per value of row_col (in row_order), one column per site 1..n_sites.

    A cell holds value_col of the variant at that (row, site), NaN where there is none.
    pivot() raises ValueError if two variants land in the same cell, so a successful call also checks that.
    """
    grid = df.pivot(index=row_col, columns="pos", values=value_col)   # long -> wide, like tidyr::pivot_wider()
    return grid.reindex(index=row_order, columns=range(1, n_sites + 1))   # force the row order and all 403 sites


saved_files = {}   # figure name -> path of the saved PDF (a dict, so re-running a cell does not add duplicates)


def save_pdf(fig, name, folder):
    """Save fig as folder/VKC_CLAUDE_<name>.pdf, record the path in saved_files, and return it."""
    path = folder / f"VKC_CLAUDE_{name}.pdf"   # Path / "text" builds the path
    fig.savefig(path, bbox_inches="tight")   # "tight" keeps labels placed outside the axes
    saved_files[name] = path
    return path


def plot_site_heatmap(grid, norm, cbar_label, title, figsize, ytick_size=7, cbar_ticks=None, extend="neither",
                      backdrop=None, undefined=None, group_labels=None, regions=None, legend_items=None):
    """Heatmap of grid (rows x sites). Color = value on the given norm; NaN = light grey.

    backdrop: 0/1 array, 1 marks cells drawn dark grey underneath (the WT residue); NaN cells then show the backdrop.
    undefined: grid with 1.0 where the value exists but is undefined (score <= 0), drawn black on top.
    group_labels: list of (label, first_row, last_row) written to the right of the plot with a white line above each block.
    regions: the region table; its boundaries are drawn as dotted lines and the names written on top.
    Returns the figure.
    """
    n_rows, n_sites = grid.shape
    cmap = plt.get_cmap("RdBu").copy()   # red = below the center (WT), blue = above; copy() so we do not change the shared colormap
    cmap.set_bad((0, 0, 0, 0) if backdrop is not None else "lightgrey")   # NaN color: transparent (RGBA alpha 0) or light grey
    # extent = (left, right, bottom, top): puts pixel columns on site numbers 1..n_sites and pixel rows on 0..n_rows-1
    extent = (0.5, n_sites + 0.5, n_rows - 0.5, -0.5)

    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    if backdrop is not None:
        ax.imshow(backdrop, aspect="auto", cmap=ListedColormap(["lightgrey", "dimgray"]), vmin=0, vmax=1,
                  interpolation="nearest", extent=extent)
    im = ax.imshow(grid.to_numpy(), aspect="auto", cmap=cmap, norm=norm, interpolation="nearest", extent=extent)
    if undefined is not None:
        # masked_invalid hides the NaNs so only the 1.0 cells are painted black
        ax.imshow(np.ma.masked_invalid(undefined.to_numpy()), aspect="auto", cmap=ListedColormap(["black"]),
                  vmin=0, vmax=1, interpolation="nearest", extent=extent)

    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(grid.index, fontsize=ytick_size)
    ax.set(xlabel="PTEN site", xlim=(0.5, n_sites + 0.5))
    ax.set_title(title, pad=18 if regions is not None else 6)   # pad = points between the plot and the title; leaves room for region names
    if regions is not None:
        for _, r in regions.iterrows():
            for x in (r["start"] - 0.5, r["end"] + 0.5):   # a tuple of the two edges of the region
                ax.axvline(x, color="black", lw=0.6, ls=":")
            # x in data units, y in axes units (1.01 = just above the plot), set by the blended transform
            ax.text((r["start"] + r["end"]) / 2, 1.01, r["region"].replace("_", " "), transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=7)
    if group_labels is not None:
        for label, first, last in group_labels:
            ax.axhline(first - 0.5, color="white", lw=1)
            ax.text(1.005, (first + last) / 2, label, transform=ax.get_yaxis_transform(), va="center", ha="left", fontsize=6)
    if legend_items is not None:
        ax.legend(handles=[Patch(facecolor=c, edgecolor="gray", label=l) for c, l in legend_items],
                  loc="upper center", bbox_to_anchor=(0.5, -0.8 / figsize[1]), ncol=len(legend_items), fontsize=7, frameon=False)
    fig.colorbar(im, ax=ax, location="bottom", shrink=0.5, pad=(1.2 if legend_items is not None else 0.8) / figsize[1], label=cbar_label, ticks=cbar_ticks, extend=extend)   # pad is a fraction of the axes height, so scale it by the figure height
    return fig


def draw_window_heatmap(values, counts, norm, cbar_label, title, ticks, extend, regions, window_size, n_sites):
    """hm_abund_3 / hm_diff_3 layout: class changes (y) by windows of window_size sites (x), n written in each cell. Returns the figure."""
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("lightgrey")
    fig, ax = plt.subplots(figsize=(13, 6.5), layout="constrained")
    im = ax.imshow(values.to_numpy(), aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_yticks(range(16))
    ax.set_yticklabels([c.replace("->", " → ") for c in values.index], fontsize=8)
    starts = [w * window_size + 1 for w in values.columns]   # first site of each window
    ax.set_xticks(range(len(starts)))
    ax.set_xticklabels([f"{s}-{min(s + window_size - 1, n_sites)}" for s in starts], rotation=45, ha="right", fontsize=7)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, str(counts.iloc[i, j]), ha="center", va="center", fontsize=5,
                    color=label_color(cmap, norm, values.iloc[i, j]))   # n in each cell
    for k in range(1, 4):
        ax.axhline(4 * k - 0.5, color="white", lw=2)
    for _, r in regions.iterrows():
        for edge in (r["start"] - 1, r["end"]):   # site edges: between site start-1 and start, and between end and end+1
            ax.axvline(edge / window_size - 0.5, color="black", lw=0.6, ls=":")
    ax.set(xlabel="PTEN sites (window)", title=title)
    fig.colorbar(im, ax=ax, location="bottom", shrink=0.5, pad=0.12, label=cbar_label, ticks=ticks, extend=extend)
    return fig


def label_color(cmap, norm, value):
    """Text color for a heatmap cell: white on dark cells, black on light ones, dim grey on NaN (grey) cells."""
    if pd.isna(value):
        return "dimgray"
    r, g, b, _ = cmap(norm(value))   # the RGBA color the heatmap uses for this value; "_" ignores the alpha
    return "white" if 0.299 * r + 0.587 * g + 0.114 * b < 0.45 else "black"   # weighted sum = perceived brightness


def cell_labels(values, counts, fmt, stars=None):
    """Text for each cell of a class-change table: the value (formatted with fmt, blank if NaN), then (n = count).

    stars: optional DataFrame of strings (such as "*") appended to the value.
    """
    labels = pd.DataFrame("", index=values.index, columns=values.columns)
    for row in values.index:
        for col in values.columns:
            value = values.loc[row, col]
            text = "" if pd.isna(value) else f"{value:{fmt}}"   # f"{x:{fmt}}": the format spec itself can be a variable
            if stars is not None:
                text += stars.loc[row, col]
            labels.loc[row, col] = f"{text}\n(n={counts.loc[row, col]})"
    return labels


def plot_class_by_region(values, labels, norm, cbar_label, title, figsize=(6.5, 8), extend="neither", cbar_ticks=None):
    """Heatmap of a (16 class changes x regions) table with the text in labels written in each cell. Returns the figure."""
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("lightgrey")   # NaN (too few variants) = grey, never a score color
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    im = ax.imshow(values.to_numpy(), aspect="auto", cmap=cmap, norm=norm)
    n_rows, n_cols = values.shape
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels([c.replace("_", " ") for c in values.columns], rotation=30, ha="right")
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels([c.replace("->", " → ") for c in values.index], fontsize=8)
    for i in range(n_rows):
        for j in range(n_cols):
            ax.text(j, i, labels.iloc[i, j], ha="center", va="center", fontsize=6,
                    color=label_color(cmap, norm, values.iloc[i, j]))   # iloc = position-based [row, column]
    for k in range(1, 4):   # a white line between the four WT-class blocks of 4 rows
        ax.axhline(4 * k - 0.5, color="white", lw=2)
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, label=cbar_label, ticks=cbar_ticks, extend=extend, shrink=0.8)
    return fig


def permutation_test(values, flag, n_perm, rng, alternative="two-sided"):
    """Is median(values[flag]) - median(values[~flag]) unusual if the flag labels were random?

    values: 1-D numpy array. flag: boolean array, True = the group of interest (needs both True and False).
    Shuffles the labels n_perm times. alternative: "two-sided", or "less" (the group of interest is LOWER).
    Returns (observed difference, p-value, array of the shuffled differences).
    The p-value is (shuffles at least as extreme + 1) / (n_perm + 1); the +1 counts the observed labeling itself.
    """
    if not (flag.any() and (~flag).any()):   # ~ is "not" for boolean arrays
        raise ValueError("flag needs both True and False values")
    observed = np.median(values[flag]) - np.median(values[~flag])   # boolean mask on an array selects elements
    null = np.empty(n_perm)   # an empty array to fill in
    for i in range(n_perm):
        shuffled = rng.permutation(flag)   # same number of True, randomly reassigned to the values
        null[i] = np.median(values[shuffled]) - np.median(values[~shuffled])
    if alternative == "less":
        extreme = null <= observed
    else:
        extreme = np.abs(null) >= abs(observed)
    return observed, (extreme.sum() + 1) / (n_perm + 1), null


def within_site_permutation_test(values, flag, sites, n_perm, rng):
    """Like permutation_test (alternative "less"), but the labels are shuffled only among variants of the SAME site.

    That keeps each site's own baseline and its WT residue fixed, so a site cannot make the group look different.
    sites: array with the site number of each value.
    """
    observed = np.median(values[flag]) - np.median(values[~flag])
    # for each site, the positions (indexes) of its variants in values; np.flatnonzero returns indexes where True
    site_indexes = [np.flatnonzero(sites == s) for s in np.unique(sites)]
    null = np.empty(n_perm)
    for i in range(n_perm):
        shuffled = flag.copy()
        for idx in site_indexes:
            shuffled[idx] = rng.permutation(flag[idx])   # reshuffle labels inside this site only
        null[i] = np.median(values[shuffled]) - np.median(values[~shuffled])
    return observed, ((null <= observed).sum() + 1) / (n_perm + 1), null

Locate the data. This notebook assumes Jupyter runs with its working directory set to this notebook's folder (`code/my_code`), so the project root is two levels up. The PDF output folder is created if it does not exist.

In [ ]:
# Find the project folders and check that the input files and output folder exist.
project_root = Path.cwd().parent.parent
data_dir = project_root / "data" / "raw_data"
pdf_dir = project_root / "data" / "my_data" / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)   # make the folder if missing, like dir.create(recursive = TRUE)

abundance_path = data_dir / "derived" / "pten-variant-abundance.csv"
regions_path = data_dir / "derived" / "pten-region-annotations.csv"
fasta_path = data_dir / "derived" / "pten-reference-protein.fasta"

abundance_path.exists(), regions_path.exists(), fasta_path.exists(), pdf_dir.exists()

In [ ]:
# Load the data and build the analysis table: one row per missense variant, with class, region, window and log-ratio columns.
abundance = pd.read_csv(abundance_path)
regions = pd.read_csv(regions_path)
ref_seq = read_fasta(fasta_path)
n_sites = len(ref_seq)

# Settings (named here so every figure uses the same values)
window_size = 25          # sites per window for hm_abund_3 / hm_diff_3
minimum_window_n = 3      # a (class change x window) median needs at least this many variants, otherwise grey
minimum_cell_n = 5        # a (class change x region) median needs at least this many variants, otherwise grey
region_order = ["phosphatase_domain", "c2_domain", "disordered_tail", "unannotated"]

missense = get_checked_missense(abundance, ref_seq)   # all 4112 missense variants; none are dropped by a site filter
site_region = make_site_region(regions, n_sites)

missense["wt_class"] = missense["wt"].map(aa_class)    # dict lookup: "His" -> "charged"
missense["mut_class"] = missense["mut"].map(aa_class)
missense["change"] = missense["wt_class"] + "->" + missense["mut_class"]   # string columns can be added together
missense["change_type"] = np.where(missense["wt_class"] == missense["mut_class"], "same class", "class-changing")   # np.where = ifelse()
missense["substitution"] = missense["wt"] + ">" + missense["mut"]   # e.g. "His>Asp": the substitution type, shared by many sites
missense["region"] = missense["pos"].map(site_region)
missense["window"] = (missense["pos"] - 1) // window_size   # // is integer division: sites 1-25 -> 0, 26-50 -> 1, ...
missense["log2_ratio"] = log2_ratio(missense["score"])

missense.head()

## 1. Checks (and what makes a check good)

A good check **can fail, and when it fails it tells you what went wrong.** Looking at `shape` or `head()` is a good habit, but it only checks what you happened to look at, once. An `assert` is the same check made permanent: it runs every time the notebook runs, and stops the notebook at the first broken assumption. Three kinds are used here:
- **Known-answer** checks: a small input where you know the right output (`log2(0.5)` must be `-1`).
- **Conservation** checks: something that must stay the same (the number of variants must still be 4112 after adding columns, and each table must sum back to 4112).
- **Consistency** checks: two different routes to the same quantity must agree (every site has one WT class).

Each check below says which plausible mistake it would catch.

In [ ]:
# Checks on the class assignment and the new columns. The comment after each assert names the mistake it would catch.

# Known-answer: the chart has 20 amino acids in classes of size 5 / 4 / 3 / 8.
assert len(aa_class) == 20                          # catches a residue missing from, or listed twice in, the dictionary
assert set(aa_class) == set(letter)                 # catches a typo such as "Tpr" for "Trp" (it would map to NaN later)
assert pd.Series(aa_class).value_counts().to_dict() == {"hydrophobic": 8, "charged": 5, "polar": 4, "special": 3}

# Conservation: adding columns must not add or lose rows. 4112 is the missense count in manifest.json.
assert len(missense) == 4112
assert missense[["wt_class", "mut_class"]].notna().all().all()   # catches a residue the class dictionary did not cover

# Consistency: a site has one WT residue, so one WT class. A mixed site would mean a WT letter was misread.
assert missense.groupby("pos")["wt_class"].nunique().max() == 1

# The 4 x 4 table of class changes: it must sum to every variant, and all 16 combinations should occur.
change_table = pd.crosstab(missense["wt_class"], missense["mut_class"]).reindex(index=class_order, columns=class_order)
assert change_table.to_numpy().sum() == len(missense)       # a missing or doubled variant would break the total
assert (change_table.to_numpy() > 0).all()                  # an empty cell would mean a class change is untestable
assert np.trace(change_table.to_numpy()) == 1002            # the diagonal = same-class variants (known answer from the data audit)
assert change_table.loc["charged", "hydrophobic"] == 552    # the cell in the hypothesis: 552 variants (known answer)
assert missense["change"].nunique() == 16

# Known-answer test of log2_ratio: 1.0 -> 0 (same as WT), 0.5 -> -1 (half of WT), 2.0 -> +1, and scores <= 0 -> NaN.
test_scores = pd.Series([1.0, 0.5, 2.0, 0.0, -0.1])
test_result = log2_ratio(test_scores)
assert test_result.iloc[0] == 0 and test_result.iloc[1] == -1 and test_result.iloc[2] == 1
assert test_result.iloc[3:].isna().all()            # a zero or negative score must be NaN, not 0 and not -inf
assert missense["log2_ratio"].isna().sum() == (missense["score"] <= 0).sum() == 16   # 16 undefined log ratios, none lost, none invented

change_table

## 2. Abundance heatmaps (raw score, WT = 1)

Same color scale in all four: diverging, centered on **WT = 1** (red below, blue above), with limits −0.25 to 1.5, the range of the data. Light grey means no measured variant.

In [ ]:
# hm_abund_0: every site (x) by every possible mutant residue (y). Messy and sparse on purpose.
# Dark grey = the WT residue at that site (no variant possible there), light grey = possible but not measured, color = score.
abund_norm = TwoSlopeNorm(vmin=-0.25, vcenter=1, vmax=1.5)   # a color scale whose midpoint (white) is at 1 = WT
abund_ticks = [-0.25, 0, 0.5, 1, 1.5]

# True where the row's residue is the WT residue of that site.
# [:, None] makes a column and [None, :] a row, so == compares every residue with every site (like outer() in R).
wt_by_site = pd.Series([one_to_three[a] for a in ref_seq], index=range(1, n_sites + 1))   # list comprehension: 1-letter -> 3-letter
wt_backdrop = (np.array(aa_order)[:, None] == wt_by_site.to_numpy()[None, :]).astype(int)

grid_abund_0 = make_grid(missense, "mut", aa_order, "score", n_sites)
assert grid_abund_0.notna().sum().sum() == len(missense)   # one filled cell per variant: no two variants share a cell, none are lost

fig = plot_site_heatmap(grid_abund_0, abund_norm, "abundance score (WT = 1)", "hm_abund_0: abundance of every measured variant",
                        figsize=(13, 5.5), cbar_ticks=abund_ticks, backdrop=wt_backdrop, regions=regions,
                        legend_items=[("dimgray", "WT residue"), ("lightgrey", "not measured")])
save_pdf(fig, "hm_abund_0_all_sites_by_mutant_residue", pdf_dir)
plt.show()

In [ ]:
# Row orders for hm_abund_1 and hm_abund_2: one row per observed substitution type (wt>mut), e.g. "His>Asp".
# Each type occurs at many sites (a column each), so unlike hgvs_pro rows, the rows are shared across sites.
subs = missense.drop_duplicates("substitution")[["substitution", "wt", "mut", "wt_class", "mut_class", "change"]]   # one row per type
aa_rank = {aa: i for i, aa in enumerate(aa_order)}              # enumerate gives (0, first), (1, second), ...
class_rank = {c: i for i, c in enumerate(class_order)}
subs = subs.assign(wt_rank=subs["wt"].map(aa_rank), mut_rank=subs["mut"].map(aa_rank),
                   wt_class_rank=subs["wt_class"].map(class_rank), mut_class_rank=subs["mut_class"].map(class_rank))

# hm_abund_1 order: by WT residue, then mutant residue (both in chart order)
order_1 = subs.sort_values(["wt_rank", "mut_rank"])["substitution"].tolist()
# hm_abund_2 order: the same rows, grouped into the 16 class changes first
subs_2 = subs.sort_values(["wt_class_rank", "mut_class_rank", "wt_rank", "mut_rank"])
order_2 = subs_2["substitution"].tolist()
group_sizes = subs_2.groupby("change", sort=False).size()   # sort=False keeps the order of first appearance

# (label, first_row, last_row) of each class-change block, for the group labels on the y-axis
group_labels = []
first_row = 0
for change, size in group_sizes.items():
    group_labels.append((change.replace("->", " → "), first_row, first_row + size - 1))
    first_row += size

assert len(order_1) == len(order_2) == subs["substitution"].nunique() == 373   # 373 observed of the 380 possible (known answer)
assert set(order_1) == set(order_2)                 # hm_abund_2 is the same rows as hm_abund_1, only reordered
assert group_sizes.sum() == len(order_2) and len(group_sizes) == 16   # every row lands in exactly one of the 16 blocks
print(len(order_1), "observed substitution types;", len(group_labels), "class-change blocks")

In [ ]:
# hm_abund_1: sites (x) by observed wt>mut substitution types (y), sorted by WT residue then mutant residue.
grid_abund_1 = make_grid(missense, "substitution", order_1, "score", n_sites)
assert grid_abund_1.notna().sum().sum() == len(missense)   # still exactly one cell per variant

fig = plot_site_heatmap(grid_abund_1, abund_norm, "abundance score (WT = 1)",
                        "hm_abund_1: abundance by observed substitution type (rows sorted by WT then mutant residue)",
                        figsize=(12, 15), ytick_size=2.6, cbar_ticks=abund_ticks, regions=regions)
save_pdf(fig, "hm_abund_1_observed_substitutions", pdf_dir)
plt.show()

In [ ]:
# hm_abund_2: the same rows and cells as hm_abund_1, but grouped by class change (WT class → mutant class). NOT aggregated:
# every colored cell is still one variant. Only the row order and the group labels differ.
grid_abund_2 = make_grid(missense, "substitution", order_2, "score", n_sites)
assert grid_abund_2.notna().sum().sum() == len(missense)             # no aggregation: still one cell per variant
assert grid_abund_2.notna().sum().sum() == grid_abund_1.notna().sum().sum()
assert np.allclose(grid_abund_2.sum(axis=1), grid_abund_1.loc[order_2].sum(axis=1))   # each row holds the same numbers, only moved

fig = plot_site_heatmap(grid_abund_2, abund_norm, "abundance score (WT = 1)",
                        "hm_abund_2: abundance grouped by class change (not aggregated)",
                        figsize=(12, 15), ytick_size=2.6, cbar_ticks=abund_ticks, group_labels=group_labels, regions=regions)
save_pdf(fig, "hm_abund_2_grouped_by_class_change", pdf_dir)
plt.show()

In [ ]:
# hm_abund_3: aggregated. For each class change and each 25-site window, the median score of the variants in it.
# Why windows and not single sites: a site has one WT class, so it can fill at most 4 of the 16 rows, usually from 1-3 variants.
group_keys = [missense["change"], missense["window"]]   # grouping by two columns = group_by(change, window)
window_median = missense.groupby(group_keys)["score"].median().unstack("window")   # unstack: long -> wide, like pivot_wider
window_n = missense.groupby(group_keys).size().unstack("window")
n_windows = (n_sites - 1) // window_size + 1
window_median = window_median.reindex(index=change_order, columns=range(n_windows))
window_n = window_n.reindex(index=change_order, columns=range(n_windows)).fillna(0).astype(int)   # empty cell = 0 variants

assert window_n.to_numpy().sum() == len(missense)       # the counts add back to all variants: nothing dropped or counted twice
assert window_n.shape == (16, n_windows) == (16, 17)
window_median = window_median.where(window_n >= minimum_window_n)   # keep medians backed by enough variants, else NaN

fig = draw_window_heatmap(window_median, window_n, abund_norm, "median abundance score (WT = 1)",
                          f"hm_abund_3: median score per class change and {window_size}-site window (numbers = variants; grey = fewer than {minimum_window_n})",
                          abund_ticks, "neither", regions, window_size, n_sites)
save_pdf(fig, "hm_abund_3_class_change_by_window", pdf_dir)
plt.show()

## 3. The same four heatmaps as a log ratio to WT

`log2(score / 1)`: 0 = same as WT, −1 = half of WT, +1 = twice WT. The scores are already relative to WT (WT = 1), so this changes the *scale* and not the information. Two consequences are visible in the figures:
- **16 variants have a score ≤ 0**, so the log ratio does not exist for them. They are drawn **black** (not 0, not dropped silently).
- **Scores near 0 are stretched.** The color scale is cut off at −3 (score 0.125), and variants below it are saturated (the arrow on the colorbar).

In [ ]:
# hm_diff_0: the hm_abund_0 layout, colored by log2(score / WT). Black = undefined log ratio (score <= 0).
diff_norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=0.6)   # white = 0 = same as WT; the data go up to log2(1.46) = 0.55
diff_ticks = [-3, -2, -1, 0, 0.5]
print("variants below the color-scale floor of -3 (score < 0.125):", int((missense["log2_ratio"] < -3).sum()))

undefined = missense[missense["score"] <= 0].assign(flag=1.0)   # the 16 variants with no log ratio; flag is the value to pivot
undefined_0 = make_grid(undefined, "mut", aa_order, "flag", n_sites)
undefined_1 = make_grid(undefined, "substitution", order_1, "flag", n_sites)
undefined_2 = make_grid(undefined, "substitution", order_2, "flag", n_sites)
assert undefined_0.notna().sum().sum() == undefined_1.notna().sum().sum() == 16   # the black cells are exactly the 16 variants

grid_diff_0 = make_grid(missense, "mut", aa_order, "log2_ratio", n_sites)
assert grid_diff_0.notna().sum().sum() == len(missense) - 16   # every variant except the 16 undefined ones is colored

legend_diff = [("dimgray", "WT residue"), ("lightgrey", "not measured"), ("black", "score <= 0 (no log ratio)")]
fig = plot_site_heatmap(grid_diff_0, diff_norm, "log2(score / WT)", "hm_diff_0: log ratio to WT of every measured variant",
                        figsize=(13, 5.5), cbar_ticks=diff_ticks, extend="min", backdrop=wt_backdrop, undefined=undefined_0,
                        regions=regions, legend_items=legend_diff)
save_pdf(fig, "hm_diff_0_all_sites_by_mutant_residue", pdf_dir)
plt.show()

In [ ]:
# hm_diff_1: the hm_abund_1 layout (observed wt>mut substitutions), colored by log2(score / WT).
grid_diff_1 = make_grid(missense, "substitution", order_1, "log2_ratio", n_sites)
assert grid_diff_1.notna().sum().sum() == len(missense) - 16

fig = plot_site_heatmap(grid_diff_1, diff_norm, "log2(score / WT)", "hm_diff_1: log ratio to WT by observed substitution type",
                        figsize=(12, 15), ytick_size=2.6, cbar_ticks=diff_ticks, extend="min", undefined=undefined_1, regions=regions,
                        legend_items=[("lightgrey", "not measured"), ("black", "score <= 0 (no log ratio)")])
save_pdf(fig, "hm_diff_1_observed_substitutions", pdf_dir)
plt.show()

In [ ]:
# hm_diff_2: the hm_abund_2 layout (grouped by class change, not aggregated), colored by log2(score / WT).
grid_diff_2 = make_grid(missense, "substitution", order_2, "log2_ratio", n_sites)
assert grid_diff_2.notna().sum().sum() == len(missense) - 16
assert grid_diff_2.notna().equals(grid_diff_1.loc[order_2].notna())   # same filled cells as hm_diff_1, only the rows moved

fig = plot_site_heatmap(grid_diff_2, diff_norm, "log2(score / WT)", "hm_diff_2: log ratio to WT grouped by class change (not aggregated)",
                        figsize=(12, 15), ytick_size=2.6, cbar_ticks=diff_ticks, extend="min", undefined=undefined_2,
                        group_labels=group_labels, regions=regions,
                        legend_items=[("lightgrey", "not measured"), ("black", "score <= 0 (no log ratio)")])
save_pdf(fig, "hm_diff_2_grouped_by_class_change", pdf_dir)
plt.show()

In [ ]:
# hm_diff_3: aggregated. For each class change and window: the median of the per-variant log ratios (log first, then summarize).
median_of_log = missense.groupby(group_keys)["log2_ratio"].median().unstack("window").reindex(index=change_order, columns=range(n_windows))

# Check: is "median of the logs" the same as "log of the median"? log2 is monotone, so the middle value is the same variant on both
# scales and the answer is YES when the number of variants is odd. With an even number the median is the average of the two middle
# values, and (log a + log b) / 2 is not log((a + b) / 2), so the two routes differ a little. Cells with a score <= 0 are left out.
log_of_median = log2_ratio(missense.groupby(group_keys)["score"].median().unstack("window").reindex(index=change_order, columns=range(n_windows)))
has_nonpositive = (missense["score"] <= 0).groupby(group_keys).any().unstack("window").reindex(index=change_order, columns=range(n_windows)).fillna(False).astype(bool)
odd_clean = ~has_nonpositive & (window_n % 2 == 1)                     # odd count, every log ratio defined
even_clean = ~has_nonpositive & (window_n % 2 == 0) & (window_n > 0)   # even count, every log ratio defined
assert np.allclose(median_of_log.where(odd_clean).to_numpy(), log_of_median.where(odd_clean).to_numpy(), equal_nan=True)   # exact for odd n
gap = (median_of_log - log_of_median).where(even_clean).abs()
print("cells with a score <= 0:", int(has_nonpositive.to_numpy().sum()), "| largest gap between the two routes in even-count cells:", round(float(np.nanmax(gap.to_numpy())), 3))

diff_window = median_of_log.where(window_n >= minimum_window_n)
assert diff_window.notna().equals(window_median.notna())   # the same cells are shown in hm_abund_3 and hm_diff_3

fig = draw_window_heatmap(diff_window, window_n, diff_norm, "median of log2(score / WT)",
                          f"hm_diff_3: median log ratio per class change and {window_size}-site window (numbers = variants)",
                          diff_ticks, "min", regions, window_size, n_sites)
save_pdf(fig, "hm_diff_3_class_change_by_window", pdf_dir)
plt.show()

In [ ]:
# What the log does to the scale: the score against its log ratio, and the linear difference (score - 1) as a heatmap for comparison.
fig, ax = plt.subplots(figsize=(5.5, 4), layout="constrained")
ax.scatter(missense["score"], missense["log2_ratio"], s=3, color="gray")   # the 16 undefined (NaN) variants are not drawn
ax.axhline(0, ls="--", color="black", lw=1)   # WT
ax.axvline(1, ls="--", color="black", lw=1)
ax.set(xlabel="abundance score (WT = 1)", ylabel="log2(score / WT)",
       title="log2 stretches low scores and compresses high ones")
save_pdf(fig, "scale_score_vs_log2_ratio", pdf_dir)
plt.show()

# hm_diff_0 again, but as the linear difference from WT (score - 1): 0 = WT, -1 = fully lost. No variant is undefined.
linear_norm = TwoSlopeNorm(vmin=-1.25, vcenter=0, vmax=0.5)
grid_linear_0 = make_grid(missense.assign(linear_diff=missense["score"] - 1), "mut", aa_order, "linear_diff", n_sites)
fig = plot_site_heatmap(grid_linear_0, linear_norm, "score - 1 (linear difference from WT)",
                        "Comparison: linear difference from WT in the hm_diff_0 layout", figsize=(13, 5.5),
                        cbar_ticks=[-1, -0.5, 0, 0.5], backdrop=wt_backdrop, regions=regions,
                        legend_items=[("dimgray", "WT residue"), ("lightgrey", "not measured")])
save_pdf(fig, "hm_diff_0_linear_difference_comparison", pdf_dir)
plt.show()

## 4. Summaries by region

Raw first, then split by class change, then the log ratio. Regions use the UniProt coordinates from the course annotation table (phosphatase 14–185, C2 190–350, disordered tail 352–403); sites in none of them are `unannotated` (kept, and labeled, but not part of the hypothesis).

In [ ]:
# box_region_abund_0: raw abundance scores by region, every variant as a dot, n in the label.
region_labels = [f"{r.replace('_', ' ')}\n(n = {(missense['region'] == r).sum()})" for r in region_order]   # list comprehension + f-string

fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
sns.boxplot(data=missense, x="region", y="score", order=region_order, color="lightgray", fliersize=0, ax=ax)   # fliersize=0: no outlier dots, the dots below show everything
sns.stripplot(data=missense, x="region", y="score", order=region_order, size=1.5, alpha=0.3, color="black", ax=ax)
ax.axhline(1, ls="--", color="gray", lw=1)   # WT
ax.set_xticks(range(len(region_order)), region_labels)
ax.set(xlabel="", ylabel="abundance score (WT = 1)", title="Abundance by region (all missense variants)")
save_pdf(fig, "box_region_abund_0_by_region", pdf_dir)
plt.show()

assert sum((missense["region"] == r).sum() for r in region_order) == len(missense)   # the regions partition the variants

In [ ]:
# box_region_abund_1: the same, split into same-class and class-changing variants (the first part of the hypothesis).
type_colors = sns.color_palette("colorblind", 2)   # two colors, colorblind-safe
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
sns.boxplot(data=missense, x="region", y="score", hue="change_type", order=region_order, hue_order=["same class", "class-changing"],
            palette=type_colors, fliersize=1, ax=ax)
ax.axhline(1, ls="--", color="gray", lw=1)
ax.set_xticks(range(len(region_order)), region_labels)
ax.set(xlabel="", ylabel="abundance score (WT = 1)", title="Abundance by region, same-class vs class-changing")
ax.legend(title=None)
save_pdf(fig, "box_region_abund_1_by_region_and_change_type", pdf_dir)
plt.show()

# the numbers behind the boxes: number of variants and median score for each combination
missense.groupby(["region", "change_type"])["score"].agg(["size", "median"]).round(3)

In [ ]:
# hm_region_abund and hm_region_diff: median score (raw), then log2 of the median, for each class change x region.
# Cells with fewer than minimum_cell_n variants are grey; the number of variants is written in every cell.
cell_keys = [missense["change"], missense["region"]]
cell_n = missense.groupby(cell_keys).size().unstack("region").reindex(index=change_order, columns=region_order).fillna(0).astype(int)
cell_median = missense.groupby(cell_keys)["score"].median().unstack("region").reindex(index=change_order, columns=region_order)
assert cell_n.to_numpy().sum() == len(missense)   # every variant is in exactly one (class change, region) cell
cell_median = cell_median.where(cell_n >= minimum_cell_n)   # NaN where too few variants

fig = plot_class_by_region(cell_median, cell_labels(cell_median, cell_n, ".2f"), abund_norm, "median abundance score (WT = 1)",
                           "hm_region_abund: median score by class change and region", cbar_ticks=abund_ticks)
save_pdf(fig, "hm_region_abund_class_change_by_region", pdf_dir)
plt.show()

# the log ratio is taken per variant first, then summarized (a score <= 0 has no log ratio and is skipped by the median)
cell_diff = missense.groupby(cell_keys)["log2_ratio"].median().unstack("region").reindex(index=change_order, columns=region_order)
cell_diff = cell_diff.where(cell_n >= minimum_cell_n)
assert cell_diff.notna().equals(cell_median.notna())   # the same cells are shown in both heatmaps
fig = plot_class_by_region(cell_diff, cell_labels(cell_diff, cell_n, "+.2f"), diff_norm, "median of log2(score / WT)",
                           "hm_region_diff: log ratio to WT by class change and region", extend="min", cbar_ticks=diff_ticks)
save_pdf(fig, "hm_region_diff_class_change_by_region", pdf_dir)
plt.show()

In [ ]:
# box_region_diff: the log ratio by region, same-class vs class-changing. seaborn silently skips NaN, so count what is skipped.
print("variants without a log ratio (score <= 0), by region:")
print(missense[missense["log2_ratio"].isna()]["region"].value_counts())

fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
sns.boxplot(data=missense, x="region", y="log2_ratio", hue="change_type", order=region_order, hue_order=["same class", "class-changing"],
            palette=type_colors, fliersize=1, ax=ax)
ax.axhline(0, ls="--", color="gray", lw=1)   # 0 = same as WT
ax.set_xticks(range(len(region_order)), region_labels)
ax.set(xlabel="", ylabel="log2(score / WT)", title="Log ratio to WT by region, same-class vs class-changing")
ax.legend(title=None)
save_pdf(fig, "box_region_diff_by_region_and_change_type", pdf_dir)
plt.show()

## 5. Testing the hypothesis

**Statistic.** Median abundance score, signed (the direction matters: loss is below WT = 1). The tests use the linear score, which keeps all 4112 variants (the log ratio would drop the 16 with a score ≤ 0). The median is nearly scale-free: the same variant is in the middle on both scales (exactly so for an odd count; see the check under `hm_diff_3`). Where a difference is reported, it is *median(target group) − median(comparison group)*, so a negative number means lower abundance in the target group.

**Why a permutation test.** It needs no assumption about the shape of the score distribution (which is not normal, see Session 5) and works for any statistic, including a median. Idea: if the label "charged → hydrophobic" had nothing to do with the score, shuffling the labels among the variants would give differences as large as the observed one fairly often. The p-value is the fraction of shuffles that are at least as extreme as what we saw.

**Three tests.**
1. *Pre-specified* (your prediction): charged → hydrophobic in the phosphatase domain versus all other phosphatase-domain variants, one-sided (predicted lower).
2. *Within-site version of test 1.* Variants at the same site share a site effect, so they are not independent. Here only charged-WT sites of the phosphatase domain are used, and "hydrophobic mutant" labels are shuffled only among the variants of the same site.
3. *Exploratory screen:* the same statistic for every class change in each annotated region (up to 48 tests), two-sided. With 48 tests a few small p-values are expected by chance, so these are for finding patterns, not for confirming them.

In [ ]:
# Known-answer tests of the two permutation functions before using them.
test_rng = np.random.default_rng(1)   # a random number generator with a fixed seed, so reruns give the same numbers

# Clear difference: 20 values at 0.1 versus 80 values at 1.0 -> observed difference -0.9 and a tiny p-value.
test_values = np.array([0.1] * 20 + [1.0] * 80)    # list * 20 repeats the element; + joins two lists
test_flag = np.array([True] * 20 + [False] * 80)
observed, p, null = permutation_test(test_values, test_flag, 2000, test_rng, alternative="less")
assert np.isclose(observed, -0.9)
assert p < 0.01        # catches a test that cannot tell a huge difference from noise
assert len(null) == 2000

# No difference: identical values -> observed 0 and p = 1 (every shuffle is "at least as extreme").
observed, p, _ = permutation_test(np.full(100, 0.7), test_flag, 500, test_rng)   # np.full(n, x) = rep(x, n)
assert observed == 0 and p == 1                    # catches a test that finds an effect in pure noise

# Within-site version: two sites; at each, the flagged value is lower than the other -> p should be small, observed -0.9.
sites_test = np.repeat(np.arange(30), 4)           # 0,0,0,0,1,1,1,1,... : 30 sites with 4 variants each
values_test = np.tile([0.1, 1.0, 1.0, 1.0], 30)    # each site: one low value followed by three high values
flag_test = np.tile([True, False, False, False], 30)
observed, p, _ = within_site_permutation_test(values_test, flag_test, sites_test, 500, test_rng)
assert np.isclose(observed, -0.9) and p < 0.01
print("permutation-test checks passed")

In [ ]:
# Tests 1 and 2: charged -> hydrophobic in the phosphatase domain.
rng = np.random.default_rng(20241005)   # fixed seed: the p-values are reproducible; a different seed changes them slightly
n_perm = 10000

phos = missense[missense["region"] == "phosphatase_domain"]
phos_values = phos["score"].to_numpy()
phos_flag = ((phos["wt_class"] == "charged") & (phos["mut_class"] == "hydrophobic")).to_numpy()   # & = "and" for boolean columns
print("phosphatase domain:", len(phos), "variants,", phos_flag.sum(), "charged -> hydrophobic")
assert phos_flag.sum() == (phos["change"] == "charged->hydrophobic").sum()   # two ways to build the flag must agree

# Test 1: against all other phosphatase variants
obs_1, p_1, null_1 = permutation_test(phos_values, phos_flag, n_perm, rng, alternative="less")
print(f"test 1: median score {np.median(phos_values[phos_flag]):.3f} vs {np.median(phos_values[~phos_flag]):.3f} for the rest;"
      f" difference {obs_1:+.3f}, one-sided p = {p_1:.4f}")

# Test 2: only sites whose WT residue is charged; compare hydrophobic mutants with the other mutants at the same sites
charged_sites = phos[phos["wt_class"] == "charged"]
site_values = charged_sites["score"].to_numpy()
site_flag = (charged_sites["mut_class"] == "hydrophobic").to_numpy()
obs_2, p_2, null_2 = within_site_permutation_test(site_values, site_flag, charged_sites["pos"].to_numpy(), n_perm, rng)
print(f"test 2: {charged_sites['pos'].nunique()} charged-WT sites, {site_flag.sum()} hydrophobic mutants vs {(~site_flag).sum()} others;"
      f" median {np.median(site_values[site_flag]):.3f} vs {np.median(site_values[~site_flag]):.3f};"
      f" difference {obs_2:+.3f}, one-sided p = {p_2:.4f}")

# Figure: where the observed difference sits among the shuffled ones
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for ax, null, observed, p, title in zip(axes, [null_1, null_2], [obs_1, obs_2], [p_1, p_2],
                                        ["Test 1: vs all other phosphatase variants", "Test 2: within charged-WT sites"]):
    ax.hist(null, bins=40, color="lightgray")
    ax.axvline(observed, color="red", lw=2, label=f"observed {observed:+.3f} (p = {p:.4f})")
    ax.set(xlabel="median difference after shuffling labels (target - rest)", ylabel="number of shuffles", title=title)
    ax.legend(fontsize=8)
save_pdf(fig, "perm_null_charged_to_hydrophobic_phosphatase", pdf_dir)
plt.show()

In [ ]:
# Test 3: exploratory screen of every class change within each annotated region (compared with the rest of that region).
screen_rows = []   # collect one dict per test, then turn the list into a table
for region in region_order[:3]:   # [:3] = the first three, the annotated regions
    in_region = missense[missense["region"] == region]
    for change in change_order:
        flag = (in_region["change"] == change).to_numpy()
        if flag.sum() < minimum_cell_n:   # too few variants to test
            continue
        observed, p, _ = permutation_test(in_region["score"].to_numpy(), flag, 5000, rng)
        screen_rows.append({"region": region, "change": change, "n": int(flag.sum()), "median_difference": observed, "p_value": p})
screen = pd.DataFrame(screen_rows)   # a list of dicts becomes one row per dict, like bind_rows()
n_tests = len(screen)
bonferroni = 0.05 / n_tests   # the cutoff if every test were a separate chance to be fooled
print(n_tests, "tests; Bonferroni cutoff", round(bonferroni, 5))
assert n_tests <= 48 and screen["p_value"].between(0, 1).all()   # at most 16 x 3 tests; p-values are probabilities

screen.sort_values("p_value").head(10).round(4)

In [ ]:
# hm_region_screen: the screen as a landscape. Color = median score of the class change minus the median of the rest of
# the region (red = lower abundance). * = p < 0.05 (uncorrected), ** = p < the Bonferroni cutoff.
diff_table = screen.pivot(index="change", columns="region", values="median_difference").reindex(index=change_order, columns=region_order[:3])
p_table = screen.pivot(index="change", columns="region", values="p_value").reindex(index=change_order, columns=region_order[:3])
n_table = screen.pivot(index="change", columns="region", values="n").reindex(index=change_order, columns=region_order[:3]).fillna(0).astype(int)

stars = pd.DataFrame("", index=diff_table.index, columns=diff_table.columns)
stars[p_table < 0.05] = "*"           # boolean DataFrame as a mask: assign where True
stars[p_table < bonferroni] = "**"
limit = np.nanmax(np.abs(diff_table.to_numpy()))   # nan-aware max: skips the untested cells

fig = plot_class_by_region(diff_table, cell_labels(diff_table, n_table, "+.2f", stars), TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit),
                           "median score: class change minus rest of region", "hm_region_screen: which class changes stand out in each region?\n* p < 0.05, ** p < Bonferroni cutoff (exploratory)",
                           figsize=(6, 8))
save_pdf(fig, "hm_region_screen_class_change_vs_rest_of_region", pdf_dir)
plt.show()

## 6. Output check

Every figure above was saved as a PDF in `data/my_data/pdfs/`. This confirms they all exist.

In [ ]:
# Check that every saved PDF exists and is not empty, then list them.
assert len(saved_files) == 17, f"expected 17 figures, found {len(saved_files)}"   # the count of save_pdf calls in this notebook
for name, path in saved_files.items():
    assert path.exists() and path.stat().st_size > 0, f"missing or empty: {path}"   # st_size = file size in bytes
    print(f"{path.stat().st_size / 1024:7.0f} KB  {path.name}")

## Presentation notes (draft, written from the results of this run; edit freely)

### Results in one paragraph
Variants in the phosphatase and C2 domains have much lower median abundance (0.72 and 0.74 of WT) than the disordered tail (0.92), and class-changing substitutions are somewhat lower than same-class ones (all regions: 0.76 vs 0.81; phosphatase 0.70 vs 0.78; C2 0.72 vs 0.77; tail 0.92 vs 0.94). The specific prediction does **not** hold as stated: charged → hydrophobic in the phosphatase domain has a median of 0.727 versus 0.717 for the rest of the domain (difference +0.009, one-sided p = 0.66). It does hold in a narrower sense: among variants at sites with a charged WT residue, hydrophobic mutants are lower than other mutants at the same sites (0.727 vs 0.827, within-site p = 0.0002). The strongest effects are in a different place: **WT hydrophobic residues replaced by charged, polar or special residues**, in the phosphatase and C2 domains (for example hydrophobic → charged, median 0.35 in the phosphatase domain and 0.38 in C2). In the screen of 48 class change × region cells, 7 pass the Bonferroni cutoff, and 6 of them are hydrophobic → charged / polar / special in those two domains.

### What I did
- Assigned every amino acid to one of four side-chain classes (from `aminoacids-pic.png`) and labeled each missense variant with its WT class, mutant class, and one of 16 class changes.
- Made heatmaps of the raw abundance score (`hm_abund_0` to `hm_abund_3`) and of the log ratio to WT (`hm_diff_0` to `hm_diff_3`), from all sites × all mutant residues down to class change × 25-site window.
- Summarized by region, split by same-class and class-changing, and tested the hypothesis with permutation tests plus an exploratory screen of all class change × region cells.

### How the code works (one explicit example for the talk)
- `aa_class` is a dictionary from residue to class, and `missense["wt"].map(aa_class)` looks up each variant's WT class; `change` is the two classes joined with `->`.
- `make_grid` pivots the long table (one row per variant) into a wide grid (rows = residue or substitution type, columns = sites 1 to 403); `plot_site_heatmap` draws the grid with one color scale centered on WT = 1.
- `permutation_test` is the central idea: compute the median difference between the target group and the rest, shuffle the group labels 10,000 times, recompute each time, and report how often a shuffle is at least as extreme as the real difference. `within_site_permutation_test` does the same but only shuffles labels among variants of the same site, so a site's own baseline cannot create the effect.

### How I checked
- Class assignment: 20 residues, class sizes 5 / 4 / 3 / 8, no unmapped residue, one WT class per site, the 4 × 4 table sums to 4112 with all 16 cells observed.
- Heatmaps: every grid has exactly one filled cell per variant (no collisions, no losses); `hm_*_2` has the same rows and filled cells as `hm_*_1`; aggregated tables sum back to all 4112 variants.
- Log ratio: known answers (`log2(0.5) = -1`), and the 16 scores ≤ 0 are NaN (drawn black), not 0 or dropped. Log of the median equals the median of the logs for odd counts but not for even counts (largest gap 0.14 in log2 units), which is why the log ratio is taken per variant first.
- Permutation tests: known-answer tests (a planted difference gives a tiny p-value, identical values give p = 1), and a fixed random seed.

### Limitations to say out loud
- Abundance only: the assay says nothing about activity or disease. "Buried hydrophobic residue" is a possible explanation consistent with these results, not something this data shows (no structure was used).
- Four classes are a crude proxy for how different a side chain is (Ala → Trp is "same class"). The WT class matters more than the mutant class here, and WT class composition differs by region (the tail has few hydrophobic WT residues).
- The screen is exploratory: 48 tests, labeled as such. Variants at the same site are not independent; test 2 handles that for one comparison, but the screen shuffles variants.
- Signed medians can hide a class change that has large effects in both directions (scores above 1 exist). The magnitude question was left for later.
- Three of the four hypothesis statements are descriptive in this notebook (class-changing vs same-class, phosphatase and C2 vs tail); only the charged → hydrophobic claim has a formal test.

## Reproducibility record

Run this last, after restarting the kernel and running all cells.

In [ ]:
import session_info
session_info.show(dependencies=True)